In [1]:
# 1. Drive 연결 · SAS 원자료에서 정의표 변수만 읽기
import os, sys, subprocess, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAS_PATH = f'{DATA_DIR}/kyrbs2025.sas7bdat'
COLS = ['INT_SPWD','INT_SPWD_TM','E_S_RCRD','SEX','GRADE','E_SES',
        'M_STR','PA_TOT','F_BR','W','STRATA','CLUSTER']

# 각 팀원은 본인 Google Drive에 원자료를 같은 경로로 보관합니다.
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')

# 아래 환경변수는 로컬에서 코드를 검증할 때만 사용 (Colab에서는 SAS 사용)
local_csv = os.environ.get('KYRBS_LOCAL_CSV')
local_sas = os.environ.get('KYRBS_LOCAL_SAS')
if Path(SAS_PATH).is_file():
    source_path = SAS_PATH
elif local_sas and Path(local_sas).is_file():
    source_path = local_sas
elif local_csv and Path(local_csv).is_file():
    source_path = local_csv
else:
    raise FileNotFoundError(f'원자료를 찾지 못했습니다: {SAS_PATH}\n개인 Drive의 hdac_data에 파일을 넣으세요.')

if str(source_path).lower().endswith('.csv'):
    raw = pd.read_csv(source_path, usecols=COLS, low_memory=False)
else:
    if importlib.util.find_spec('pyreadstat') is None and Path('/content').exists():
        subprocess.check_call([sys.executable,'-m','pip','-q','install','pyreadstat'])
    if importlib.util.find_spec('pyreadstat') is not None:
        import pyreadstat
        raw, _ = pyreadstat.read_sas7bdat(source_path, usecols=COLS)
    else:
        raw = pd.read_sas(source_path, format='sas7bdat', encoding='cp949')[COLS].copy()

for col in COLS:
    if col not in ['STRATA','CLUSTER']:
        raw[col] = pd.to_numeric(raw[col],errors='coerce')
raw['STRATA'] = raw['STRATA'].astype(str)
raw['PSU_KEY'] = raw['STRATA'] + '|' + raw['CLUSTER'].astype(str)
SOURCE = '질병관리청 제21차(2025년) 청소년건강행태조사 원시자료'
print('원자료:',source_path)
print('자료 크기:',raw.shape,'| 층:',raw.STRATA.nunique(),'| 층 내 PSU:',raw.PSU_KEY.nunique())

Mounted at /content/drive
원자료: /content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat
자료 크기: (54170, 13) | 층: 106 | 층 내 PSU: 796


In [2]:
# 2. 10~11주차와 동일한 노출·결과·공변량·in_analysis 및 완전 사례
raw['phone_min'] = np.where(raw['INT_SPWD'].eq(1), 0.0, raw['INT_SPWD_TM'])
raw['eligible'] = raw['GRADE'].isin(range(1,7))
raw['exposure_ok'] = raw['phone_min'].ge(0) & np.isfinite(raw['phone_min'])
raw['outcome_ok'] = raw['E_S_RCRD'].isin([1,2,3,4,5])
raw['in_analysis'] = raw['eligible'] & raw['exposure_ok'] & raw['outcome_ok']
base_vars = ['SEX','GRADE','E_SES']
extra_vars = ['M_STR','PA_TOT','F_BR']
raw['complete_case'] = (raw['in_analysis'] & raw[base_vars].notna().all(axis=1)
    & raw['SEX'].isin([1,2]) & raw['GRADE'].isin(range(1,7))
    & raw['E_SES'].isin([1,2,3,4,5]) & raw['W'].gt(0))
raw['high_grade'] = raw['E_S_RCRD'].isin([1,2]).astype(int)
raw['phone_hr'] = raw['phone_min']/60

raw['phone_group'] = pd.Series(index=raw.index,dtype='object')
raw.loc[raw.phone_min.eq(0),'phone_group'] = '미사용'
raw.loc[raw.phone_min.gt(0)&raw.phone_min.lt(120),'phone_group'] = '0분 초과–2시간 미만'
raw.loc[raw.phone_min.ge(120)&raw.phone_min.lt(240),'phone_group'] = '2–4시간 미만'
raw.loc[raw.phone_min.ge(240)&raw.phone_min.lt(360),'phone_group'] = '4–6시간 미만'
raw.loc[raw.phone_min.ge(360),'phone_group'] = '6시간 이상'
GROUP_ORDER=['미사용','0분 초과–2시간 미만','2–4시간 미만','4–6시간 미만','6시간 이상']

# 추가 보정은 S1에서만 사용, 결측이 있다면 같은 완전 사례 중 교집합을 선택
for col, valid in [('M_STR',range(1,6)),('PA_TOT',range(1,9)),('F_BR',range(1,9))]:
    raw[f'{col}_valid'] = raw[col].isin(valid)
raw['s1_complete_case'] = raw['complete_case'] & raw[[f'{c}_valid' for c in extra_vars]].all(axis=1)

n_raw=len(raw); n_t1=int(raw.in_analysis.sum()); n_base=int(raw.complete_case.sum());n_s1=int(raw.s1_complete_case.sum())
print(f'원자료 {n_raw:,}명 | Table 1 {n_t1:,}명 | 11주차/12주차 주 분석 {n_base:,}명 | S1 {n_s1:,}명')
print(f'Table 1과 주 분석 차이: {n_t1-n_base:,}명 (경제상태 등 보정변수 결측 여부 점검)')
print('집단별 공통 완전 사례:')
display(raw.loc[raw.complete_case,'phone_group'].value_counts().reindex(GROUP_ORDER).rename('n').to_frame())
assert n_t1==54164, 'Table 1의 54,164명과 불일치: 원자료 및 노출/결과 코딩을 확인하세요.'
assert n_base==54158, '11주차 완전 사례 54,158명과 불일치: 결측 코딩을 확인하세요.'
assert n_s1==n_base, 'S1에 추가 결측이 있습니다. S1 대상자와 주 분석 차이를 각주에 기록하세요.'
assert raw.loc[raw.complete_case,'phone_group'].notna().all()

원자료 54,170명 | Table 1 54,164명 | 11주차/12주차 주 분석 54,158명 | S1 54,158명
Table 1과 주 분석 차이: 6명 (경제상태 등 보정변수 결측 여부 점검)
집단별 공통 완전 사례:


,n
phone_group,
미사용,1688
0분 초과–2시간 미만,4865
2–4시간 미만,18203
4–6시간 미만,16167
6시간 이상,13235


In [3]:
# 3. 복합표본설계 기반 회귀 함수 (11주차와 같은 설계·도메인 처리)
import statsmodels.api as sm
from scipy.stats import t as tdist

psu_info = raw[['STRATA','PSU_KEY']].drop_duplicates().reset_index(drop=True)
psu_info['psu_idx']=np.arange(len(psu_info))
psu_codes=raw['PSU_KEY'].map(psu_info.set_index('PSU_KEY')['psu_idx']).to_numpy(dtype=int)
strata_groups=[g['psu_idx'].to_numpy(dtype=int) for _,g in psu_info.groupby('STRATA',sort=False)]
if any(len(ix)<2 for ix in strata_groups):
    raise ValueError('PSU 1개인 층이 있어 추가 분산추정 규칙 검토가 필요합니다.')
design_df = len(psu_info)-len(strata_groups)
print('설계 자유도 (PSU 수 - 층 수):',design_df)

def fit_svy(xdf, term_names, family='logit', domain_col='complete_case'):
    mask=raw[domain_col].to_numpy(dtype=bool)
    Xfull=sm.add_constant(xdf.astype(float),has_constant='add')
    X=Xfull.loc[mask].to_numpy(dtype=float)
    y=raw.loc[mask,'high_grade'].to_numpy(dtype=float)
    w0=raw.loc[mask,'W'].to_numpy(dtype=float)
    w=w0/w0.mean()  # 공통 상수 재척도: 계수 및 샌드위치 공분산 불변
    fam=sm.families.Binomial() if family=='logit' else sm.families.Poisson()
    res=sm.GLM(y,X,family=fam,freq_weights=w).fit(maxiter=180)
    mu=np.asarray(res.predict(X))
    if family=='logit':
        hwt=w*mu*(1-mu)
    else:
        hwt=w*mu
    H=(X.T*hwt)@X
    Hinv=np.linalg.inv(H)
    # 도메인 밖의 기여는 0이며 원자료의 모든 PSU와 층을 유지
    scores=np.zeros((len(raw),X.shape[1]),dtype=float)
    scores[mask,:]=(w*(y-mu))[:,None]*X
    psu_scores=np.zeros((len(psu_info),X.shape[1]),dtype=float)
    np.add.at(psu_scores,psu_codes,scores)
    meat=np.zeros_like(H)
    for ix in strata_groups:
        u=psu_scores[ix,:]
        centered=u-u.mean(axis=0)
        meat+=(len(ix)/(len(ix)-1))*(centered.T@centered)
    cov=Hinv@meat@Hinv
    se=np.sqrt(np.diag(cov)); b=np.asarray(res.params)
    crit=tdist.ppf(.975,design_df)
    p=2*tdist.sf(np.abs(b/se),design_df)
    return pd.DataFrame({'변수':['절편']+list(term_names),
        '추정치':np.exp(b),'하한':np.exp(b-crit*se),'상한':np.exp(b+crit*se),
        'p':p,'beta':b,'SE_beta':se})

def dummy_vars(col, levels):
    return pd.DataFrame({f'{col}_{v}':raw[col].eq(v).astype(float)
                         for v in levels[1:]}, index=raw.index)

X_base=pd.concat([
    raw[['phone_hr']],dummy_vars('SEX',[1,2]),dummy_vars('GRADE',[1,2,3,4,5,6]),
    dummy_vars('E_SES',[1,2,3,4,5])],axis=1)

def row_of(result,var):
    return result.loc[result['변수']==var].iloc[0]

def value_ci(row):
    return f"{row['추정치']:.3f} ({row['하한']:.3f}–{row['상한']:.3f})"

def fmt_p(p):
    return '<0.0001' if p<.0001 else f'{p:.4f}'

설계 자유도 (PSU 수 - 층 수): 690


In [4]:
# 4. 주 분석 재현: 11주차 모형 2 (성별·학년·경제상태 보정)
main_result=fit_svy(X_base,list(X_base.columns))
main_row=row_of(main_result,'phone_hr')
print('11주차 주 분석 재현:',value_ci(main_row),'| p=',fmt_p(main_row.p),'| n=',n_base)
assert abs(main_row['추정치']-0.8543973)<0.002, '11주차 OR 0.854와 차이 발생: 코딩 및 설계 점검'

11주차 주 분석 재현: 0.854 (0.845–0.864) | p= <0.0001 | n= 54158


In [5]:
# 5. S1: 스트레스·신체활동·아침식사 빈도를 추가 보정
X_s1=pd.concat([X_base,
    dummy_vars('M_STR',[1,2,3,4,5]),
    dummy_vars('PA_TOT',[1,2,3,4,5,6,7,8]),
    dummy_vars('F_BR',[1,2,3,4,5,6,7,8])],axis=1)
s1_result=fit_svy(X_s1,list(X_s1.columns),domain_col='s1_complete_case')
s1_row=row_of(s1_result,'phone_hr')
print('S1 결과:',value_ci(s1_row),'| p=',fmt_p(s1_row.p),'| n=',n_s1)
print('주 분석과 OR 차이:',f"{s1_row['추정치']-main_row['추정치']:+.3f}")

S1 결과: 0.864 (0.855–0.873) | p= <0.0001 | n= 54158
주 분석과 OR 차이: +0.010


In [6]:
# 6. S3: 사용시간을 5개 범주로 변경 (미사용 기준)
X_cat=pd.DataFrame(index=raw.index)
for group in GROUP_ORDER[1:]:
    X_cat[group]=raw.phone_group.eq(group).astype(float)
X_s3=pd.concat([X_cat,X_base.drop(columns=['phone_hr'])],axis=1)
s3_result=fit_svy(X_s3,list(X_s3.columns))
rows=[{'구간':'미사용','OR (95% CI)':'1.000 (기준)','p값':'—','n':n_base}]
for group in GROUP_ORDER[1:]:
    r=row_of(s3_result,group)
    rows.append({'구간':group,'OR (95% CI)':value_ci(r),'p값':fmt_p(r.p),'n':n_base})
s3_table=pd.DataFrame(rows)
display(s3_table)

,구간,OR (95% CI),p값,n
0,미사용,1.000 (기준),—,54158
1,0분 초과–2시간 미만,1.473 (1.274–1.705),<0.0001,54158
2,2–4시간 미만,0.955 (0.835–1.091),0.4960,54158
3,4–6시간 미만,0.618 (0.539–0.708),<0.0001,54158
4,6시간 이상,0.396 (0.345–0.455),<0.0001,54158


In [7]:
# 7. S-PR: 동일한 보정변수, 로그 연결 포아송 회귀 + 설계기반 샌드위치 분산
pr_result=fit_svy(X_base,list(X_base.columns),family='poisson')
pr_row=row_of(pr_result,'phone_hr')
print('S-PR 결과:',value_ci(pr_row),'| p=',fmt_p(pr_row.p),'| n=',n_base)

S-PR 결과: 0.911 (0.905–0.917) | p= <0.0001 | n= 54158


In [8]:
# 8. D-2 Table 3 출력 및 주 분석 차이 표시
rows=[
    ['주 분석(11주차 모형 2)','1시간 증가',n_base,'OR',value_ci(main_row),fmt_p(main_row.p),'기준'],
    ['S1. 추가 보정','1시간 증가',n_s1,'OR',value_ci(s1_row),fmt_p(s1_row.p),f"OR {s1_row['추정치']-main_row['추정치']:+.3f}"],
    ['S3. 노출 범주화','미사용',n_base,'OR','1.000 (기준)','—','기준 범주']
]
for group in GROUP_ORDER[1:]:
    r=row_of(s3_result,group)
    if group=='0분 초과–2시간 미만':
        comparison=f"반대 방향: 오즈 {(r['추정치']-1)*100:.1f}% 높음"
    elif group=='2–4시간 미만':
        comparison='낮은 방향이나 유의하지 않음' if r['하한']<=1<=r['상한'] else '주 분석과 방향 비교'
    else:
        comparison=f"같은 방향: 오즈 {(1-r['추정치'])*100:.1f}% 낮음"
    rows.append(['S3. 노출 범주화',group,n_base,'OR',value_ci(r),fmt_p(r.p),comparison])
rows.append(['S-PR. 유병률비','1시간 증가',n_base,'PR',value_ci(pr_row),fmt_p(pr_row.p),'효과측정 지표 변경'])
table3=pd.DataFrame(rows,columns=['분석','노출변수 또는 비교 구간','분석 인원(n)','지표','추정치 (95% CI)','p값','주 분석과의 차이'])
display(Markdown('## Table 3. 주 분석 및 민감도분석 결과'))
display(table3)
print(f'Table 1 {n_t1:,}명 → Table 3 {n_base:,}명: 경제상태 결측 {n_t1-n_base:,}명 제외')
print('가중치 W / 층 STRATA / 집락 CLUSTER | 도메인 밖 점수 0 | Taylor 선형화 | FPC 미적용')

## Table 3. 주 분석 및 민감도분석 결과

,분석,노출변수 또는 비교 구간,분석 인원(n),지표,추정치 (95% CI),p값,주 분석과의 차이
0,주 분석(11주차 모형 2),1시간 증가,54158,OR,0.854 (0.845–0.864),<0.0001,기준
1,S1. 추가 보정,1시간 증가,54158,OR,0.864 (0.855–0.873),<0.0001,OR +0.010
2,S3. 노출 범주화,미사용,54158,OR,1.000 (기준),—,기준 범주
3,S3. 노출 범주화,0분 초과–2시간 미만,54158,OR,1.473 (1.274–1.705),<0.0001,반대 방향: 오즈 47.3% 높음
4,S3. 노출 범주화,2–4시간 미만,54158,OR,0.955 (0.835–1.091),0.4960,낮은 방향이나 유의하지 않음
5,S3. 노출 범주화,4–6시간 미만,54158,OR,0.618 (0.539–0.708),<0.0001,같은 방향: 오즈 38.2% 낮음
6,S3. 노출 범주화,6시간 이상,54158,OR,0.396 (0.345–0.455),<0.0001,같은 방향: 오즈 60.4% 낮음
7,S-PR. 유병률비,1시간 증가,54158,PR,0.911 (0.905–0.917),<0.0001,효과측정 지표 변경


Table 1 54,164명 → Table 3 54,158명: 경제상태 결측 6명 제외
가중치 W / 층 STRATA / 집락 CLUSTER | 도메인 밖 점수 0 | Taylor 선형화 | FPC 미적용


In [9]:
# 9. Table 3 Markdown 및 CSV 저장 (원자료는 저장/공유하지 않음)
OUT=Path('/content/12week_outputs') if Path('/content').exists() else Path('/mnt/data/12week_outputs')
OUT.mkdir(parents=True,exist_ok=True)
report=f"""# D-2. Table 3 — 민감도분석 결과

{table3.to_markdown(index=False)}

## 분석 각주

1. **자료 이름 및 연도:** {SOURCE}.
2. **복합표본설계:** 가중치 `W`, 층화변수 `STRATA`, 집락변수 `CLUSTER`. 층 내 PSU 중심화에 의한 Taylor 선형화 분산추정, FPC 미적용.
3. **주 분석:** 스마트폰 사용시간 1시간 증가당 OR; 높은 학업성적(상·중상=1)에 대한 이분형 로지스틱 회귀, 성별·학년·경제상태 보정.
4. **S1 변경점:** 스트레스 인지(`M_STR`), 신체활동(`PA_TOT`), 아침식사 빈도(`F_BR`)를 범주형 보정변수로 추가.
5. **S3 변경점:** 연속형 사용시간을 5개 구간으로 변경. 미사용군 기준. 각 행의 n={n_base:,}명은 범주별 인원이 아닌 전체 회귀모형 분석 인원.
6. **S-PR 변경점:** 동일한 설명변수에 로그 연결 포아송 회귀를 적용하여 유병률비(PR)를 산출. 설계기반 샌드위치 분산추정.
7. **결측 및 도메인:** Table 1 {n_t1:,}명에서 경제상태 결측 {n_t1-n_base:,}명을 제외한 주 분석 {n_base:,}명, S1 {n_s1:,}명. 도메인 밖의 점수 기여는 0으로 두고 층·집락 구조를 보존.
8. **통계 지표:** OR(오즈비), PR(유병률비), 95% CI, 양측 p값. 설계 자유도 {design_df}.
9. **주 분석과의 차이:** S1은 OR 척도에서 차이를 제시. S3는 미사용군 대비 각 구간의 OR과 주 분석의 방향을 비교하며 노출 단위가 달라 OR의 수치 차이는 계산하지 않음. S-PR은 지표가 달라 방향을 중심으로 비교.
"""
(OUT/'12_Table3.md').write_text(report,encoding='utf-8')
table3.to_csv(OUT/'12_Table3.csv',index=False,encoding='utf-8-sig')
print('저장 위치:',OUT)
print('생성:',OUT/'12_Table3.md',OUT/'12_Table3.csv')
display(Markdown(report))

저장 위치: /content/12week_outputs
생성: /content/12week_outputs/12_Table3.md /content/12week_outputs/12_Table3.csv


# D-2. Table 3 — 민감도분석 결과

| 분석                   | 노출변수 또는 비교 구간   |   분석 인원(n) | 지표   | 추정치 (95% CI)     | p값     | 주 분석과의 차이            |
|:-----------------------|:--------------------------|---------------:|:-------|:--------------------|:--------|:----------------------------|
| 주 분석(11주차 모형 2) | 1시간 증가                |          54158 | OR     | 0.854 (0.845–0.864) | <0.0001 | 기준                        |
| S1. 추가 보정          | 1시간 증가                |          54158 | OR     | 0.864 (0.855–0.873) | <0.0001 | OR +0.010                   |
| S3. 노출 범주화        | 미사용                    |          54158 | OR     | 1.000 (기준)        | —       | 기준 범주                   |
| S3. 노출 범주화        | 0분 초과–2시간 미만       |          54158 | OR     | 1.473 (1.274–1.705) | <0.0001 | 반대 방향: 오즈 47.3% 높음  |
| S3. 노출 범주화        | 2–4시간 미만              |          54158 | OR     | 0.955 (0.835–1.091) | 0.4960  | 낮은 방향이나 유의하지 않음 |
| S3. 노출 범주화        | 4–6시간 미만              |          54158 | OR     | 0.618 (0.539–0.708) | <0.0001 | 같은 방향: 오즈 38.2% 낮음  |
| S3. 노출 범주화        | 6시간 이상                |          54158 | OR     | 0.396 (0.345–0.455) | <0.0001 | 같은 방향: 오즈 60.4% 낮음  |
| S-PR. 유병률비         | 1시간 증가                |          54158 | PR     | 0.911 (0.905–0.917) | <0.0001 | 효과측정 지표 변경          |

## 분석 각주

1. **자료 이름 및 연도:** 질병관리청 제21차(2025년) 청소년건강행태조사 원시자료.
2. **복합표본설계:** 가중치 `W`, 층화변수 `STRATA`, 집락변수 `CLUSTER`. 층 내 PSU 중심화에 의한 Taylor 선형화 분산추정, FPC 미적용.
3. **주 분석:** 스마트폰 사용시간 1시간 증가당 OR; 높은 학업성적(상·중상=1)에 대한 이분형 로지스틱 회귀, 성별·학년·경제상태 보정.
4. **S1 변경점:** 스트레스 인지(`M_STR`), 신체활동(`PA_TOT`), 아침식사 빈도(`F_BR`)를 범주형 보정변수로 추가.
5. **S3 변경점:** 연속형 사용시간을 5개 구간으로 변경. 미사용군 기준. 각 행의 n=54,158명은 범주별 인원이 아닌 전체 회귀모형 분석 인원.
6. **S-PR 변경점:** 동일한 설명변수에 로그 연결 포아송 회귀를 적용하여 유병률비(PR)를 산출. 설계기반 샌드위치 분산추정.
7. **결측 및 도메인:** Table 1 54,164명에서 경제상태 결측 6명을 제외한 주 분석 54,158명, S1 54,158명. 도메인 밖의 점수 기여는 0으로 두고 층·집락 구조를 보존.
8. **통계 지표:** OR(오즈비), PR(유병률비), 95% CI, 양측 p값. 설계 자유도 690.
9. **주 분석과의 차이:** S1은 OR 척도에서 차이를 제시. S3는 미사용군 대비 각 구간의 OR과 주 분석의 방향을 비교하며 노출 단위가 달라 OR의 수치 차이는 계산하지 않음. S-PR은 지표가 달라 방향을 중심으로 비교.
